In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

## LER OS RETORNOS, SCORE e SELIC

In [108]:
basedados_ibov = Path('../../base_dados/retorno_ibov_2015_2026.csv') 

df_ibov=pd.read_csv(basedados_ibov).set_index(['Date']).fillna(0)

dfs_ibov = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    nome = f'df_ibov_retornos_{ano_usado}.csv'
    caminho = Path('retornos_ibov_otimizacao') / nome
    dfs_ibov[ano_usado] = pd.read_csv(caminho).set_index('Date')

In [144]:
dfs_ibov['2015']

,IBOV
Date,
2015-10-01,0.005637
2015-10-02,0.037958
2015-10-05,0.012013
2015-10-06,0.002878
2015-10-07,0.024699
...,...
2016-03-24,-0.000664
2016-03-28,0.023783
2016-03-29,0.006235


In [146]:
dfs_ibov['2015'].iloc[-1,0]

np.float64(-0.0232980155710355)

In [80]:
df_ibov.loc['2015-01-05']

IBOV   -0.02051
Name: 2015-01-05, dtype: float64

In [105]:
df_ibov.index.star(['2015-01-05'])

array([ True, False, False, ..., False, False, False], shape=(2849,))

In [4]:
# Retornos


dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    nome = f'df_ativos_{ano_usado}.csv'
    caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv(caminho).set_index('date')
    

78

In [5]:
## SELIC para Sigma e excesso
selic_d = pd.read_csv('selic/selic_diario.csv').set_index('date')

## PIOTROSKI

In [6]:
## SCORE para score
score_todos_anos_pio = pd.read_csv('score_piotroski/piotroski.csv').set_index('Unnamed: 0').reset_index()

In [7]:
score_todos_anos_pio.rename(columns={'Unnamed: 0': 'date'}, inplace=True)

In [8]:
score_todos_anos_pio.set_index('date', inplace=True)

In [87]:
lista_ativos_finais = score_todos_anos_pio.columns.tolist()
score_todos_anos_pio

np.float64(0.375)

In [185]:
score_todos_anos_pio.describe()

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
count,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,...,11.000000,11.000000,11.000000,11.000000,11.000000,11.00000,11.000000,11.000000,11.000000,11.000000
mean,0.534091,0.693182,0.534091,0.556818,0.522727,0.579545,0.579545,0.386364,0.386364,0.465909,...,0.511364,0.602273,0.613636,0.511364,0.590909,0.62500,0.488636,0.625000,0.681818,0.613636
std,0.137964,0.116775,0.311704,0.141019,0.229253,0.225504,0.115552,0.142023,0.142023,0.113067,...,0.152628,0.255062,0.205050,0.162544,0.262743,0.30104,0.298195,0.201556,0.204356,0.212533
min,0.375000,0.500000,0.125000,0.375000,0.125000,0.000000,0.375000,0.125000,0.125000,0.250000,...,0.375000,0.125000,0.125000,0.250000,0.125000,0.12500,0.000000,0.375000,0.375000,0.250000
25%,0.375000,0.625000,0.312500,0.500000,0.375000,0.562500,0.500000,0.312500,0.312500,0.375000,...,0.375000,0.437500,0.562500,0.437500,0.437500,0.43750,0.375000,0.437500,0.500000,0.500000
50%,0.625000,0.625000,0.500000,0.500000,0.500000,0.625000,0.625000,0.375000,0.375000,0.500000,...,0.500000,0.750000,0.625000,0.500000,0.625000,0.62500,0.500000,0.625000,0.625000,0.625000
75%,0.625000,0.750000,0.812500,0.687500,0.687500,0.750000,0.625000,0.500000,0.500000,0.500000,...,0.562500,0.750000,0.687500,0.562500,0.812500,0.87500,0.687500,0.812500,0.875000,0.812500
max,0.750000,0.875000,1.000000,0.750000,0.875000,0.750000,0.750000,0.625000,0.625000,0.625000,...,0.875000,0.875000,0.875000,0.875000,0.875000,1.00000,0.875000,0.875000,1.000000,0.875000


In [95]:
score_todos_anos_pio[score_todos_anos_pio.index.isin([2016])].iloc[0,0]

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,
2016,0.625,0.75,0.375,0.75,0.75,0.0,0.5,0.5,0.5,0.625,...,0.375,0.875,0.5,0.375,0.25,1.0,0.0,0.875,0.875,0.5


## Magic Formula

In [124]:
## SCORE para score
score_todos_anos_mf = pd.read_csv('score_mf/magic_formula.csv').set_index('Unnamed: 0').reset_index()

In [125]:
score_todos_anos_mf.rename(columns={'Unnamed: 0': 'date'}, inplace=True)

In [126]:
score_todos_anos_mf.set_index('date', inplace=True)

In [127]:
# lista_ativos_finais = score_todos_anos.columns.tolist()
score_todos_anos_mf = score_todos_anos_mf[score_todos_anos_pio.columns]
score_todos_anos_mf

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,
2015,0.884615,0.307692,0.641026,0.064103,0.692308,0.897436,0.307692,0.307692,0.307692,0.012821,...,0.948718,0.307692,0.769231,0.730769,0.051282,0.076923,0.307692,0.666667,0.564103,0.307692
2016,0.897436,0.243590,0.538462,0.833333,0.666667,0.602564,0.243590,0.243590,0.243590,1.000000,...,0.923077,0.243590,0.782051,0.679487,0.089744,0.551282,0.243590,0.730769,0.576923,0.243590
2017,0.910256,0.185897,0.551282,0.461538,0.628205,0.858974,0.185897,0.185897,0.185897,0.961538,...,0.820513,0.602564,0.576923,0.564103,0.371795,0.589744,0.185897,0.666667,0.474359,0.185897
2018,0.871795,0.173077,0.320513,0.756410,0.576923,0.884615,0.173077,0.173077,0.173077,0.961538,...,0.897436,0.538462,0.551282,0.346154,0.333333,0.602564,0.173077,0.730769,0.512821,0.173077
2019,0.833333,0.198718,0.448718,0.551282,0.679487,0.923077,0.198718,0.198718,0.198718,1.000000,...,0.807692,0.525641,0.512821,0.320513,0.333333,0.294872,0.198718,0.666667,0.602564,0.730769
2020,0.782051,0.179487,0.269231,0.525641,0.730769,0.794872,0.179487,0.179487,0.179487,0.961538,...,0.935897,0.500000,0.641026,0.358974,0.589744,0.602564,0.179487,0.564103,0.538462,0.307692
2021,0.717949,0.102564,0.269231,0.538462,0.794872,0.756410,0.102564,0.102564,0.102564,0.974359,...,0.987179,0.025641,0.384615,0.243590,0.807692,0.782051,0.410256,0.448718,0.487179,0.320513
2022,0.756410,0.115385,0.371795,0.307692,0.666667,0.743590,0.115385,0.115385,0.115385,0.987179,...,0.833333,0.025641,0.384615,0.615385,0.538462,0.717949,0.551282,0.448718,0.564103,0.410256
2023,0.794872,0.935897,0.538462,0.448718,0.628205,0.769231,0.096154,0.096154,0.096154,0.987179,...,0.833333,0.166667,0.384615,0.589744,0.179487,0.564103,0.730769,0.423077,0.576923,0.461538


In [128]:
print(len(score_todos_anos_pio.columns))
print(len(score_todos_anos_mf.columns))

75
75


In [130]:
lista_ativos_finais = score_todos_anos_mf.columns.tolist()


## EXCESSO DOS ANOS e SIGMA (MAtriz de covariancia)

##### EXCESSO PARA TODOS

In [9]:
dict_sigma = {}
dict_excesso = {}
for an in anos:
    ano = an.split("-")[0]

    try:
        print(f"=============== \n EXCESSO {ano}\n ============")
        print("Atualização, Ano: ",ano)
        df = dfs[ano]
        # df_f = pd.DataFrame(eval(df))
        df_f = df.copy()
        slc = selic_d[selic_d.index.isin(df_f.index)]
        slc['valor_diario'] = slc['valor_diario']/100

        print(f"Tamanho DF de {ano}: ", len(df_f))
        print(f"Tamanho Selic: ", len(slc['valor_diario']))
        ano = int(ano)
        dict_excesso[ano] = df_f.sub(slc['valor_diario'],axis=0)
        print("tamanho final do Excesso: ",len(dict_excesso[ano]))

        print(f"============\n SIGMA {ano}\n===========")            
        dict_sigma[ano] = dict_excesso[ano].cov()
        print('Tamanho final do SIGMA: ',len(dict_sigma[ano]))

    except Exception as e:
        print(e)
        print("ERror")

 EXCESSO 2025
Atualização, Ano:  2025
Tamanho DF de 2025:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 SIGMA 2025
Tamanho final do SIGMA:  78
 EXCESSO 2024
Atualização, Ano:  2024
Tamanho DF de 2024:  122
Tamanho Selic:  122
tamanho final do Excesso:  122
 SIGMA 2024
Tamanho final do SIGMA:  78
 EXCESSO 2023
Atualização, Ano:  2023
Tamanho DF de 2023:  121
Tamanho Selic:  121
tamanho final do Excesso:  121
 SIGMA 2023
Tamanho final do SIGMA:  78
 EXCESSO 2022
Atualização, Ano:  2022
Tamanho DF de 2022:  124
Tamanho Selic:  124
tamanho final do Excesso:  124
 SIGMA 2022
Tamanho final do SIGMA:  78
 EXCESSO 2021
Atualização, Ano:  2021
Tamanho DF de 2021:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 SIGMA 2021
Tamanho final do SIGMA:  78
 EXCESSO 2020
Atualização, Ano:  2020
Tamanho DF de 2020:  120
Tamanho Selic:  120
tamanho final do Excesso:  120
 SIGMA 2020
Tamanho final do SIGMA:  78
 EXCESSO 2019
Atualização, Ano:  2019
Tamanho DF de 2019:  123
Tamanho Selic

## Hiperparâmetros

In [71]:
vb_cardinalidade_max = 10
vb_cardinalidade_min = 10
vb_peso_maximo = 0.20
vb_peso_minimo = 0.02
vb_theta = 0.5

### Fazendo otimização ano a ano e comparando com o proximo ano

In [72]:
anos = ['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']
print(anos)

['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']


In [ ]:
# score_usado = score_todos_anos[score_todos_anos.index.isin(['2015'])]
# score_usado

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,


In [210]:
from pydoc import text


y = []

carteiras_anuais = {}

melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])

print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)

for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass

    

    try:

        
        

        ano_um = int(ano)
        # df_usado = f'df_ativos_{ano_um}'
        # retorno_usado = pd.DataFrame(eval(df_usado))
        df_usado = dfs[str(ano_um)][lista_ativos_finais]
        retorno_usado = df_usado.copy()
        # retorno_usado = retorno_usado[lista_ativos_finais]
        print("Retornos atualizados")

        df_ibov_usado = dfs_ibov[str(ano_um)]
        retorno_ibov = df_ibov_usado.copy()

        # sigma_usado = dict_sigma[str(ano_um)]
        # print("Sigmas Atualizados")
        score_pio_final = score_todos_anos_pio[score_todos_anos_pio.index.isin([ano_um])]
        score_mf_final = score_todos_anos_mf[score_todos_anos_mf.index.isin([ano_um])]
        print("-----")
    except Exception as e:
        print("ERRRRRRRRRRRROR")
        print(e)

    # if df_usado == 'df_ativos_2015':
    #     continue
    # else:
    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",int(ano)+1)

    # # print("## UTILIZANDO SCORE DO ANO DE: ",ano)
    # print("## UTILIZANDO DADOS DE RETORNO DE: ",str(ano_um))
    # # print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    # print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)

    model = pyo.ConcreteModel()

    #---------VARIÁVEIS-----------
    # model.nome_ativos = pyo.Set(initialize = retorno_usado.columns.tolist())
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns.tolist())-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.retornos_ibov = pyo.Param(model.dias, initialize=lambda model,dia: retorno_ibov.iloc[dia,0])
    # model.theta = pyo.Param(initialize=vb_theta)
    model.score_p = pyo.Param(model.ativos, initialize=lambda model,a: score_pio_final.iloc[0,a])
    model.score_m = pyo.Param(model.ativos, initialize=lambda model,a: score_mf_final.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos, bounds=(0,1))
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    # model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    # model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    # model.s = pyo.Param(initialize = 2, mutable=True)
    # model.r = pyo.Var(within=pyo.NonNegativeReals)
    
    for i in range(3):
        setattr(model,f'obj{i}_mais',pyo.Var(bounds=(0,0.1),domain=NonNegativeReals))
        setattr(model,f'obj{i}_menos',pyo.Var(domain=NonNegativeReals))
    #-------------------------------------- FUNÇÕES

    #=============================
    # Função Objetivo
    #=============================

    def func_objetivo_1(model):
       
        return model.obj1_menos + model.obj2_menos
    model.objetivo = pyo.Objective(rule=func_objetivo_1, sense=pyo.minimize)

    #=============================
    # RESTRIÇÕES
    #=============================

    # # OBJ 1 2 3 - Rest 1 2 3
    # def retorno_restr_rule(model,diad):
    #     return sum(model.x[a]*model.retornos_ativos[dia,a] for a in model.ativos for dia in model.dias) + model.obj0_menos - model.obj0_mais >= model.retornos_ibov[diad]
    # model.retorno_restr = pyo.Constraint(model.dias,rule=retorno_restr_rule)

    def piotroski_restr_rule(model):
        return sum(model.x[a]*model.score_p[a] for a in model.ativos) + model.obj1_menos - model.obj1_mais >= 0.9
    model.piotroski_rest = pyo.Constraint(rule=piotroski_restr_rule)

    def mf_restr_rule(model):
        return sum(model.x[a]*model.score_m[a] for a in model.ativos) + model.obj2_menos - model.obj2_mais >= 0.9
    model.mf_restr = pyo.Constraint(rule=mf_restr_rule)

    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)

    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)

    #peso minimo por acao
    def rule_peso_minimo(model, a):
        return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)

    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)


    def cardinalidade_min(model):
        return sum(
            model.y[a] for a in model.ativos
            ) >= model.cardinalidade_valor_min
    model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)

    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)


    # NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # PC
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)


    melhor_pesos = {a: pyo.value(model.x[a]) for a in model.ativos}
    pio = {a: sum(pyo.value(model.x[a])*pyo.value(model.score_p[a]) for a in model.ativos)}
    mf = {a: sum(pyo.value(model.x[a])*pyo.value(model.score_m[a]) for a in model.ativos)}
    carteiras_anuais[int(ano_um)+1] = {
        'pesos':  melhor_pesos,
        # 'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano_um} -> {melhor_pesos}")
    print(f"{ano_um} Piotroski -> {pio}")
    # print(f"Valores obj Menos: {pyo.value(model.obj0_menos), pyo.value(model.obj1_menos), pyo.value(model.obj2_menos)}")
    print(f"Valores obj Menos: {pyo.value(model.obj1_menos), pyo.value(model.obj1_mais)}")
    print(f"{ano_um} Magic Formula -> {mf}")
    # print(f"Valores obj Menos: {pyo.value(model.obj0_menos), pyo.value(model.obj1_menos), pyo.value(model.obj2_menos)}")
    print(f"Valores obj Menos: {pyo.value(model.obj2_menos), pyo.value(model.obj2_mais)}")
    print("+-="*30)
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass




=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2015
Nao consta model
Retornos atualizados
-----
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2016
2015 -> {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.02, 6: 0.0, 7: 0.0, 8: 0.0, 9: 0.0, 10: 0.0, 11: 0.0, 12: 0.0, 13: 0.0, 14: 0.0, 15: 0.0, 16: 0.02, 17: 0.0, 18: 0.0, 19: 0.2, 20: 0.0, 21: 0.0, 22: 0.0, 23: 0.0, 24: 0.0, 25: 0.0, 26: 0.0, 27: 0.0, 28: 0.0, 29: 0.0, 30: 0.0, 31: 0.0, 32: 0.09999999999999994, 33: 0.0, 34: 0.0, 35: 0.0, 36: 0.0, 37: 0.2, 38: 0.0, 39: 0.0, 40: 0.0, 41: 0.0, 42: 0.0, 43: 0.02, 44: 0.0, 45: 0.0, 46: 0.0, 47: 0.0, 48: 0.0, 49: 0.02, 50: 0.0, 51: 0.0, 52: 0.0, 53: 0.0, 54: 0.0, 55: 0.02, 56: 0.0, 57: 0.0, 58: 0.0, 59: 0.0, 60: 0.0, 

In [206]:
carteiras_anuais

{2016: {'pesos': {0: 0.0,
   1: 0.0,
   2: 0.0,
   3: 0.0,
   4: 0.0,
   5: 0.02,
   6: 0.0,
   7: 0.0,
   8: 0.0,
   9: 0.0,
   10: 0.0,
   11: 0.0,
   12: 0.0,
   13: 0.0,
   14: 0.0,
   15: 0.0,
   16: 0.02,
   17: 0.0,
   18: 0.0,
   19: 0.2,
   20: 0.0,
   21: 0.0,
   22: 0.0,
   23: 0.0,
   24: 0.0,
   25: 0.0,
   26: 0.0,
   27: 0.0,
   28: 0.0,
   29: 0.0,
   30: 0.0,
   31: 0.0,
   32: 0.09999999999999994,
   33: 0.0,
   34: 0.0,
   35: 0.0,
   36: 0.0,
   37: 0.2,
   38: 0.0,
   39: 0.0,
   40: 0.0,
   41: 0.0,
   42: 0.0,
   43: 0.02,
   44: 0.0,
   45: 0.0,
   46: 0.0,
   47: 0.0,
   48: 0.0,
   49: 0.02,
   50: 0.0,
   51: 0.0,
   52: 0.0,
   53: 0.0,
   54: 0.0,
   55: 0.02,
   56: 0.0,
   57: 0.0,
   58: 0.0,
   59: 0.0,
   60: 0.0,
   61: 0.0,
   62: 0.0,
   63: 0.0,
   64: 0.0,
   65: 0.20000000000000007,
   66: 0.0,
   67: 0.0,
   68: 0.2,
   69: 0.0,
   70: 0.0,
   71: 0.0,
   72: 0.0,
   73: 0.0,
   74: 0.0}},
 2017: {'pesos': {0: 0.0,
   1: 0.0,
   2: 0.0,
   3: 0.

In [207]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])+1
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        if v >= vb_peso_minimo:
            linhas.append({'ano': an, 'ativo': lista_ativos_finais[k], 'peso': round(v, 4)})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [208]:
df_portfolios

,ano,ativo,peso
0,2016,B3SA3,0.02
1,2016,COGN3,0.02
2,2016,CSAN3,0.20
3,2016,FLRY3,0.10
4,2016,ISAE4,0.20
...,...,...,...
105,2026,IGTI11,0.02
106,2026,JHSF3,0.20
107,2026,MULT3,0.02
108,2026,SBSP3,0.20


In [209]:
df_portfolios.to_csv('carteiras_gp.csv')
